# 03 — OSM Feature Scope Validation

This notebook validates the candidate OpenStreetMap feature scope defined in `02_define_feature_scope.ipynb` against the actual source dataset.

The objective is not to manually inspect every selected OSM object or mapping.
Instead, the candidate scope is profiled systematically to identify
classifications that may require additional investigation due to structural,
semantic, or data-quality inconsistencies.

The validation follows a risk-oriented approach. General characteristics of the
selected features are first measured across the dataset, after which targeted
analysis is performed only for mappings presenting potentially problematic
patterns.

Examples include unusually large object populations, missing descriptive
attributes, ambiguous primary classifications, unexpected geometry types,
overlapping tagging conventions, or classifications whose usefulness depends
on secondary OSM tags.

The results of this stage may lead to candidate mappings being retained,
refined through additional filtering rules, or excluded from the final ColMaps
dataset.

## 1. Validation Strategy

The candidate scope defined in the previous stage contains multiple ColMaps
categories constructed from explicit OSM `key=value` mappings.

Validating every individual OSM object manually would not provide a scalable or
reproducible methodology. Instead, this notebook uses dataset-level profiling
to identify mappings that exhibit characteristics requiring further analysis.

The validation process is divided into three stages:

1. **General profiling** — measure the distribution and basic characteristics
   of objects belonging to the candidate feature scope.
2. **Risk identification** — identify mappings with potentially ambiguous,
   inconsistent, unusually broad, or structurally problematic characteristics.
3. **Targeted validation** — investigate only the mappings identified during
   profiling and determine whether they should be retained, refined, or
   excluded.

This approach preserves a broad semantic candidate scope while allowing final
filtering decisions to be supported by observed characteristics of the actual
Colombia OSM dataset.

## 2. Load Candidate Feature Scope

The candidate feature scope produced by
`02_define_feature_scope.ipynb` is loaded from its exported mapping file.

Using a version-controlled mapping artifact avoids duplicating the semantic
selection rules across notebooks and ensures that the validation stage operates
on exactly the same candidate scope defined during the previous stage.

In [1]:
from pathlib import Path
import pandas as pd

CANDIDATE_PBF_PATH = Path("../filtered/01_candidate_scope.osm.pbf")
FEATURE_SCOPE_PATH = Path("../filters/01_feature_scope.csv")

if not CANDIDATE_PBF_PATH.exists():
    raise FileNotFoundError(
        f"Candidate OSM extract not found: {CANDIDATE_PBF_PATH.resolve()}"
    )

if not FEATURE_SCOPE_PATH.exists():
    raise FileNotFoundError(
        f"Feature scope not found: {FEATURE_SCOPE_PATH.resolve()}"
    )

feature_scope = pd.read_csv(FEATURE_SCOPE_PATH)

print(f"Candidate OSM extract: {CANDIDATE_PBF_PATH.name}")
print(
    f"Extract size: "
    f"{CANDIDATE_PBF_PATH.stat().st_size / (1024 ** 2):.2f} MiB"
)
print(f"Candidate mappings: {len(feature_scope):,}")
print(f"ColMaps categories: {feature_scope['category'].nunique()}")
print(f"OSM keys: {feature_scope['osm_key'].nunique()}")

feature_scope.head()

Candidate OSM extract: 01_candidate_scope.osm.pbf
Extract size: 25.82 MiB
Candidate mappings: 115
ColMaps categories: 12
OSM keys: 8


,category,osm_key,osm_value,include,reason
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...


## 3. Candidate Scope Profiling

In this section we are gonna answers those 3 necesary questions:

- `How Many?`
- `Do they have name?`
- `Which form is the data?`

     
The candidate feature scope is profiled against the raw OSM dataset before any
targeted validation is performed.

The objective of this stage is to obtain general quantitative characteristics
for each selected `key=value` mapping and use them to identify classifications
that may require additional investigation.

The first profiling metric is the number of OSM objects associated with each
candidate mapping. Large differences in object frequency can reveal broad
classifications, potential outliers, or mappings whose practical suitability
should be examined more closely.

The profiling process is performed automatically from the candidate feature
scope rather than through manually defined queries, ensuring that all mappings
are evaluated consistently.

### 3.1 Object Frequency

Answers `How many?`

The first profiling metric evaluates the number of OSM objects associated with
each candidate `key=value` mapping.

Object frequency provides a general view of the scale and distribution of the
candidate feature scope. Some mappings may represent highly common services or
broad geographic classifications, while others may correspond to rare but
semantically relevant features.

The candidate mappings are evaluated automatically against the raw OSM dataset
using Osmium. All selected tag expressions are processed in a single scan of
the source `.osm.pbf` file in order to avoid repeatedly reading the complete
dataset for each individual mapping.

The resulting counts are merged with the candidate feature scope and used to
identify frequency outliers and understand the overall distribution of the
selected classifications.

Object frequency is treated as a profiling indicator rather than an inclusion
or exclusion criterion. A high number of objects does not necessarily indicate
that a mapping is too broad, while a low number of objects does not imply that
the corresponding feature type is irrelevant.



In [2]:
import subprocess

tag_expressions = (
    feature_scope["osm_key"]
    + "="
    + feature_scope["osm_value"]
).tolist()

result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=name-asc",
        str(CANDIDATE_PBF_PATH),
        *tag_expressions,
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

count_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    count_rows.append({
        "osm_key": key.strip('"'),
        "osm_value": value.strip('"'),
        "object_count": int(count),
    })

mapping_counts = pd.DataFrame(count_rows)

In [3]:
## Counting and Validation
mapping_counts.count()

osm_key         115
osm_value       115
object_count    115
dtype: int64

In [4]:
## Visualization
mapping_counts.head(50)

,osm_key,osm_value,object_count
0,amenity,bar,2461
1,amenity,bus_station,1037
2,amenity,cafe,4146
3,amenity,car_wash,608
4,amenity,casino,280
5,amenity,clinic,1157
6,amenity,doctors,874
7,amenity,fast_food,4348
8,amenity,fuel,4325
9,amenity,hospital,2196


In [5]:
## Merging and showing Max. Values
feature_profile = feature_scope.merge(
    mapping_counts,
    on=["osm_key", "osm_value"],
    how="left",
)

feature_profile["object_count"] = (
    feature_profile["object_count"]
    .fillna(0)
    .astype(int)
)

feature_profile.sort_values(
    "object_count",
    ascending=False,
).head(20)

,category,osm_key,osm_value,include,reason,object_count
26,nature,natural,water,True,Water bodies may represent scenic natural dest...,42569
59,parks_recreation,leisure,park,True,Parks represent recreational green spaces pote...,14898
68,food_drink,amenity,restaurant,True,Restaurants represent relevant meal stops for ...,14815
111,transport_travel,amenity,parking,True,Parking facilities may be useful when stopping...,10130
65,parks_recreation,leisure,swimming_pool,True,Swimming pools represent recreational faciliti...,9553
27,nature,natural,wetland,True,Wetlands may represent natural areas of ecolog...,8208
28,nature,natural,peak,True,Mountain peaks may represent scenic or explora...,7159
93,markets_shopping,shop,convenience,True,Convenience stores provide practical supplies ...,6955
74,accommodation,tourism,hotel,True,Hotels represent explicit traveler accommodation.,5655
94,markets_shopping,shop,supermarket,True,Supermarkets provide practical food and travel...,5601


In [6]:
## Distribution 
count_stats = feature_profile["object_count"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count_stats

count      115.000000
mean      1660.321739
std       4706.017985
min          1.000000
25%         33.000000
50%        241.000000
75%        951.500000
90%       4338.800000
95%       7473.700000
99%      14886.380000
max      42569.000000
Name: object_count, dtype: float64

In [7]:
## Checking Min. extremes
print(
    f"Mappings with 0 objects: "
    f"{(feature_profile['object_count'] == 0).sum()}"
)

print(
    f"Mappings with <= 10 objects: "
    f"{(feature_profile['object_count'] <= 10).sum()}"
)

print(
    f"Mappings with <= 100 objects: "
    f"{(feature_profile['object_count'] <= 100).sum()}"
)

Mappings with 0 objects: 0
Mappings with <= 10 objects: 17
Mappings with <= 100 objects: 41


In [8]:
feature_profile.sort_values(
    "object_count",
    ascending=True,
).head(20)

,category,osm_key,osm_value,include,reason,object_count
53,nature,natural,dune,True,Dunes may represent distinctive natural landsc...,1
54,nature,natural,geyser,True,Geysers represent distinctive natural phenomen...,1
55,nature,natural,peninsula,True,Peninsulas may represent distinctive geographi...,1
52,nature,natural,islet,True,Islets may represent distinctive natural desti...,2
25,history_heritage,man_made,ceremonial_gate,True,Ceremonial gates may represent culturally or h...,3
51,nature,natural,sinkhole,True,Sinkholes may represent distinctive geological...,3
50,nature,natural,gulf,True,Gulfs may represent significant scenic coastal...,3
49,nature,natural,strait,True,Straits may represent distinctive scenic geogr...,4
80,accommodation,tourism,cabin,True,Cabins represent visitor accommodation potenti...,4
48,nature,natural,waterfall,True,Waterfalls represent distinctive natural attra...,5


#### Candidate mapping distribution observations

The candidate mappings exhibit a strongly uneven frequency distribution. While
the median mapping contains 241 occurrences, a small number of broad
classifications contain several thousand objects, with the largest mapping
reaching more than 42,000 occurrences.

Low-frequency mappings are also present, particularly among specific natural
and heritage features. Their limited occurrence does not by itself indicate a
data-quality problem, as rare classifications may still represent semantically
valid and relevant places.

Consequently, object frequency is treated as a profiling indicator rather than
an inclusion or exclusion criterion. High or low occurrence counts may trigger
additional investigation when combined with semantic ambiguity or other
structural characteristics, but no mapping is discarded solely on the basis of
its frequency.

### 3.2 Name Availability

Answers `Do they have name?`

Object frequency alone is not sufficient to determine whether a candidate
mapping is suitable for ColMaps. A classification may contain many valid
features, while another mapping with a similar frequency may primarily
represent objects that are difficult to identify or present meaningfully to a
traveler.

The availability of the OSM `name` tag is therefore evaluated as an additional
profiling indicator.

For each candidate mapping, the analysis measures the number and proportion of
matching OSM objects that provide an explicit `name` value. This metric can help
identify classifications containing large populations of unnamed objects or
features whose primary OSM classification alone may not provide sufficient
information for presentation in the application.

Name availability is not treated as a strict inclusion criterion. Some
geographic features may remain useful even when unnamed, while other feature
types are expected to provide a recognizable destination or service name.
Instead, the resulting ratios are used together with object frequency and
semantic characteristics to prioritize mappings for targeted validation.


In [9]:
from pyrosm import OSM

candidate_filter = (
    feature_scope.groupby("osm_key")["osm_value"]
    .apply(list)
    .to_dict()
)

candidate_filter

{'amenity': ['theatre',
  'restaurant',
  'cafe',
  'fast_food',
  'bar',
  'ice_cream',
  'pub',
  'nightclub',
  'casino',
  'marketplace',
  'pharmacy',
  'hospital',
  'clinic',
  'doctors',
  'fuel',
  'parking',
  'parking_entrance',
  'bus_station',
  'car_wash'],
 'historic': ['monument',
  'memorial',
  'archaeological_site',
  'ruins',
  'castle',
  'city_gate',
  'citywalls',
  'battlefield',
  'wayside_shrine',
  'wayside_cross',
  'church',
  'cannon',
  'wreck',
  'manor'],
 'leisure': ['nature_reserve',
  'park',
  'garden',
  'picnic_table',
  'fishing',
  'resort',
  'golf_course',
  'swimming_pool',
  'sauna',
  'horse_riding',
  'water_park'],
 'man_made': ['lighthouse',
  'observatory',
  'obelisk',
  'windmill',
  'watermill',
  'ceremonial_gate'],
 'natural': ['water',
  'wetland',
  'peak',
  'cliff',
  'bare_rock',
  'beach',
  'ridge',
  'spring',
  'cape',
  'reef',
  'rock',
  'glacier',
  'bay',
  'cave_entrance',
  'stone',
  'desert',
  'volcano',
  'hot_s

In [10]:
%%time

osm = OSM(
    str(CANDIDATE_PBF_PATH),
    engine="out_of_core",
    workers=1,
)

candidate_features = osm.get_data_by_custom_criteria(
    custom_filter=candidate_filter,
    tags_as_columns=[
        "name",
        *candidate_filter.keys(),
    ],
    keep_nodes=True,
    keep_ways=True,
    keep_relations=True,
    keep_other_tags=False,
)

print(f"Candidate objects loaded: {len(candidate_features):,}")
candidate_features.head()

Candidate objects loaded: 188,510
CPU times: total: 39.6 s
Wall time: 40.1 s


,id,lon,lat,visible,version,timestamp,changeset,name,amenity,historic,leisure,man_made,natural,shop,tourism,waterway,geometry,osm_type
0,108685320,-71.116730,12.034026,False,5,1786922350,0.0,NaN,NaN,NaN,NaN,NaN,cape,NaN,NaN,NaN,POINT (-71.11673 12.03403),node
1,110059874,-71.312162,12.356847,False,6,1747624230,0.0,Cabo Falso,NaN,NaN,NaN,NaN,cape,NaN,NaN,NaN,POINT (-71.31216 12.35685),node
2,113195014,-76.817289,8.488190,False,3,1780181279,0.0,Boca Caño La Mesa,NaN,NaN,NaN,NaN,bay,NaN,NaN,NaN,POINT (-76.81729 8.48819),node
3,128093966,-77.808402,7.145339,False,5,1745029504,0.0,Punta Ardita,NaN,NaN,NaN,NaN,cape,NaN,NaN,NaN,POINT (-77.8084 7.14534),node
4,128257149,-77.674116,6.956651,False,3,1745197597,0.0,Punta Melo,NaN,NaN,NaN,NaN,cape,NaN,NaN,NaN,POINT (-77.67412 6.95665),node


The reduced candidate OSM extract is processed with Pyrosm using the
out-of-core engine.

A single worker is used explicitly in the notebook to avoid multiprocessing
issues associated with Jupyter on Windows. After reducing the source dataset to
the candidate feature scope, the remaining processing time is sufficiently low
for interactive validation.

Parallel workers may provide additional performance improvements in standalone
Python execution, but are not required for the current notebook workflow.


In [11]:
name_stats = []

for row in feature_scope.itertuples():
    matches = candidate_features[
        candidate_features[row.osm_key] == row.osm_value
    ]

    object_count = len(matches)

    named_count = matches["name"].notna().sum()

    named_ratio = (
        named_count / object_count
        if object_count > 0
        else 0
    )

    name_stats.append({
        "osm_key": row.osm_key,
        "osm_value": row.osm_value,
        "named_count": named_count,
        "named_ratio": named_ratio,
    })

name_profile = pd.DataFrame(name_stats)

In [12]:
feature_profile = feature_profile.merge(
    name_profile,
    on=["osm_key", "osm_value"],
    how="left",
)

feature_profile["named_percentage"] = (
    feature_profile["named_ratio"] * 100
).round(1)

In [13]:
feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "named_count",
        "named_percentage",
    ]
].sort_values(
    "named_percentage",
    ascending=True,
).head(20)

,category,osm_key,osm_value,object_count,named_count,named_percentage
25,history_heritage,man_made,ceremonial_gate,3,0,0.0
52,nature,natural,islet,2,0,0.0
61,parks_recreation,leisure,picnic_table,116,0,0.0
41,nature,natural,desert,63,0,0.0
55,nature,natural,peninsula,1,0,0.0
112,transport_travel,amenity,parking_entrance,1045,22,2.1
29,nature,natural,cliff,3674,120,3.3
60,parks_recreation,leisure,garden,4805,178,3.7
27,nature,natural,wetland,8208,303,4.2
65,parks_recreation,leisure,swimming_pool,9553,479,5.0


In [14]:
feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "named_count",
        "named_percentage",
    ]
].sort_values(
    "named_percentage",
    ascending=False,
).head(20)

,category,osm_key,osm_value,object_count,named_count,named_percentage
15,history_heritage,historic,battlefield,5,5,100.0
50,nature,natural,gulf,3,3,100.0
19,history_heritage,historic,cannon,7,7,100.0
42,nature,natural,volcano,59,59,100.0
49,nature,natural,strait,4,4,100.0
53,nature,natural,dune,1,1,100.0
54,nature,natural,geyser,1,1,100.0
80,accommodation,tourism,cabin,4,4,100.0
38,nature,natural,bay,117,114,97.4
44,nature,natural,saddle,28,27,96.4


#### Name availability observations

Name availability varies substantially across the candidate feature mappings.
Several high-frequency classifications contain only a small proportion of named
objects. Examples include `natural=water`, `natural=wetland`,
`leisure=swimming_pool`, `leisure=garden`, and `amenity=parking`.

In contrast, several destination-oriented mappings exhibit high name
availability. Museums, hospitals, clinics, aquariums, theme parks, and natural
peaks contain names for more than 90% of their observed objects. The
`natural=peak` mapping is particularly informative: despite containing more
than 7,000 objects, approximately 93% are named. This demonstrates that high
object frequency alone does not imply an overly broad or unsuitable
classification.

Percentages derived from very small populations must also be interpreted with
care. Several rare mappings reach 100% name availability while containing only
a handful of objects, making their percentages less informative than similarly
high ratios observed across larger populations.

Low name availability does not have the same significance for every feature
type. Objects such as picnic tables, parking facilities, or certain natural
features may remain meaningful without an explicit OSM `name`.

Name availability is therefore retained as a contextual profiling signal rather
than a universal quality threshold. Its relevance is considered together with
object frequency, feature semantics, geometry, and additional OSM tags. The
combined profiling results are subsequently used to prioritize mappings for
targeted validation rather than to automatically include or exclude features.

### 3.3 Geometry and OSM Object Structure

Answers `Which form is the data?`

Candidate features in OpenStreetMap are not represented uniformly. Depending on
the feature and mapping practices, a selected classification may be stored as
an OSM node, way, or relation and may consequently produce point, linear, or
polygonal geometries.

This distinction is relevant to ColMaps because the application is intended to
identify places and services in relation to a calculated route. Point features
can be represented directly as locations, while linear or polygonal features
may require an appropriate representative location or additional spatial
processing in later stages.

The candidate dataset is therefore profiled according to both its original OSM
object type and its resulting geometry type. The objective is to identify
mappings with unusual or heterogeneous structural characteristics that may
require additional consideration during targeted validation.

Geometry type is not treated as an inclusion criterion. Linear and polygonal
features may represent valid destinations, particularly for natural areas,
parks, heritage sites, and other geographic features.


In [15]:
# Count how candidate features are represented
# by the original OpenStreetMap object type.
osm_type_distribution = (
    candidate_features["osm_type"]
    .value_counts()
    .rename_axis("osm_type")
    .reset_index(name="object_count")
)

osm_type_distribution

,osm_type,object_count
0,way,110519
1,node,74334
2,relation,3657


In [16]:
# Count the geometry types produced by Pyrosm
# after reconstructing the candidate OSM features.
geometry_distribution = (
    candidate_features.geometry.geom_type
    .value_counts()
    .rename_axis("geometry_type")
    .reset_index(name="object_count")
)

geometry_distribution

,geometry_type,object_count
0,Polygon,109417
1,Point,74334
2,MultiLineString,4107
3,MultiPolygon,541
4,LineString,111


#### General structure observations

The candidate dataset contains OSM nodes, ways, and relations, confirming that
the selected feature scope is structurally heterogeneous.

After geometry reconstruction, the dataset is dominated by point and polygon
geometries, while linear and multipolygon geometries occur considerably less
frequently.

This general distribution shows that ColMaps candidate features cannot be
assumed to be exclusively point-based POIs. Some selected features represent
spatial areas or linear geographic structures, which may require different
handling in later processing stages.


#### Geometry distribution by mapping

The general geometry distribution describes the candidate dataset as a whole,
but individual OSM mappings may exhibit substantially different spatial
representations.

To identify these differences, geometry types are counted separately for each
candidate `key=value` mapping. This allows mappings that are predominantly
point-based, linear, polygonal, or structurally heterogeneous to be identified
without assuming a uniform spatial representation across the candidate scope.


In [17]:
geometry_stats = []

# Profile the geometry representation of each candidate mapping.
for row in feature_scope.itertuples():

    # Select candidate objects matching the current key=value mapping.
    matches = candidate_features[
        candidate_features[row.osm_key] == row.osm_value
    ]

    # Count the geometry types observed for this mapping.
    geometry_counts = (
        matches.geometry.geom_type
        .value_counts()
        .to_dict()
    )

    geometry_stats.append({
        "osm_key": row.osm_key,
        "osm_value": row.osm_value,
        "point_count": geometry_counts.get("Point", 0),
        "linestring_count": geometry_counts.get("LineString", 0),
        "multilinestring_count": geometry_counts.get("MultiLineString", 0),
        "polygon_count": geometry_counts.get("Polygon", 0),
        "multipolygon_count": geometry_counts.get("MultiPolygon", 0),
        "geometry_types": len(geometry_counts),
    })

geometry_profile = pd.DataFrame(geometry_stats)

geometry_profile.head()

,osm_key,osm_value,point_count,linestring_count,multilinestring_count,polygon_count,multipolygon_count,geometry_types
0,tourism,attraction,951,5,8,227,2,5
1,man_made,lighthouse,74,0,0,10,0,2
2,man_made,observatory,6,0,0,11,0,2
3,tourism,viewpoint,1006,0,0,34,0,2
4,tourism,museum,215,0,0,172,0,2


In [18]:
# Add geometry statistics to the existing candidate feature profile.
feature_profile = feature_profile.merge(
    geometry_profile,
    on=["osm_key", "osm_value"],
    how="left",
)

In [19]:
feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "point_count",
        "linestring_count",
        "multilinestring_count",
        "polygon_count",
        "multipolygon_count",
        "geometry_types",
    ]
].head(10)

,category,osm_key,osm_value,object_count,point_count,linestring_count,multilinestring_count,polygon_count,multipolygon_count,geometry_types
0,attractions,tourism,attraction,1194,951,5,8,227,2,5
1,attractions,man_made,lighthouse,84,74,0,0,10,0,2
2,attractions,man_made,observatory,17,6,0,0,11,0,2
3,viewpoints,tourism,viewpoint,1040,1006,0,0,34,0,2
4,museums_culture,tourism,museum,390,215,0,0,172,0,2
5,museums_culture,tourism,gallery,45,37,0,0,8,0,2
6,museums_culture,tourism,artwork,1022,940,13,8,60,1,5
7,museums_culture,amenity,theatre,394,148,0,0,245,0,2
8,history_heritage,historic,monument,667,566,2,0,97,0,3
9,history_heritage,historic,memorial,758,654,1,0,103,0,3


In [20]:
# Show mappings with the greatest geometry diversity.
feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "point_count",
        "linestring_count",
        "multilinestring_count",
        "polygon_count",
        "multipolygon_count",
        "geometry_types",
    ]
].sort_values(
    ["geometry_types", "object_count"],
    ascending=[False, False],
).head(20)

,category,osm_key,osm_value,object_count,point_count,linestring_count,multilinestring_count,polygon_count,multipolygon_count,geometry_types
26,nature,natural,water,42569,54,10,52,41870,112,5
27,nature,natural,wetland,8208,146,1,1,7009,30,5
0,attractions,tourism,attraction,1194,951,5,8,227,2,5
6,museums_culture,tourism,artwork,1022,940,13,8,60,1,5
68,food_drink,amenity,restaurant,14815,12073,0,3,2736,3,4
111,transport_travel,amenity,parking,10130,1310,0,1,8799,14,4
65,parks_recreation,leisure,swimming_pool,9553,160,0,3,9370,10,4
60,parks_recreation,leisure,garden,4805,145,0,1,4643,2,4
29,nature,natural,cliff,3674,59,61,3466,46,0,4
107,health_pharmacy,amenity,hospital,2196,1009,0,1,1180,5,4


#### Geometry and structure observations

The geometry profile shows that candidate mappings exhibit substantially
different spatial representations. Some mappings are predominantly point-based,
while others are primarily represented as areas or linear geographic
structures.

Several destination- and service-oriented mappings are represented mainly as
points. For example, `tourism=attraction` and `tourism=artwork` contain a large
majority of point geometries. Other mappings, such as `amenity=restaurant`,
combine point and polygon representations, reflecting different OSM mapping
practices for the same type of real-world place.

Area-oriented features exhibit a different pattern. `natural=water`,
`leisure=park`, `leisure=swimming_pool`, `leisure=garden`, and
`amenity=parking` are represented predominantly as polygons. Natural geographic
features may also exhibit strongly linear representations; for example,
`natural=cliff` is represented primarily through `MultiLineString` geometries.

The presence of multiple geometry types within a mapping does not by itself
indicate inconsistency. A feature may legitimately be represented differently
depending on its physical extent and the level of detail used by OSM
contributors. Consequently, geometry diversity is interpreted together with
the dominant geometry representation and the semantics of each mapping.

These results confirm that the candidate scope cannot be treated uniformly as
a collection of point-based POIs. Geometry and OSM object structure are
therefore retained as contextual profiling signals for the subsequent targeted
validation stage rather than being used as automatic inclusion or exclusion
criteria.


## 4. Risk Identification

The profiling results obtained from object frequency, name availability, and
geometry structure provide complementary information about the candidate
feature mappings. However, none of these characteristics is sufficient on its
own to determine whether a mapping should be retained or excluded.

The purpose of this stage is therefore to identify candidate mappings that
require additional investigation before the final feature scope is established.

A mapping may be prioritized for targeted validation when its observed
characteristics suggest potential semantic ambiguity, unusually broad
classification, limited descriptive information, structural characteristics
that may complicate its use as a route-oriented destination, or dependence on
additional OSM tags for determining its practical relevance.

Risk identification does not represent an automatic filtering procedure.
Mappings identified in this stage remain part of the candidate scope until
their characteristics are examined in the targeted validation stage.

### 4.1 Validation Risk Criteria

Candidate mappings are prioritized for targeted validation according to the
following characteristics:

* **Semantic breadth** — the OSM classification may represent a wide range of
  real-world features, not all of which are necessarily relevant to the ColMaps
  use case.

*  **Semantic relevance** — the classification is valid in OSM, but its role as a route-oriented destination or traveler service in ColMaps is uncertain.

* **Limited descriptive information** — a mapping contains a substantial
  population of objects without explicit names or other immediately useful
  descriptive information. The significance of this characteristic depends on
  the semantics of the feature type.

* **Spatial representation** — the mapping is predominantly represented through
  linear or extensive area geometries whose use as route-oriented destinations
  may require additional consideration.

* **Heterogeneous representation** — the same mapping appears through multiple
  geometry types or mapping practices that may represent meaningfully different
  real-world objects.

* **Overlapping classification** — the same or closely related real-world
  feature may be represented through multiple OSM tagging conventions within
  the candidate scope.

* **Secondary-tag dependency** — the primary `key=value` classification may not
  provide sufficient information to determine whether an object is relevant to
  ColMaps, requiring additional OSM attributes to distinguish useful features
  from unrelated or unsuitable objects.

These criteria are used as investigation triggers rather than quantitative
exclusion thresholds. A mapping may exhibit one or more of these
characteristics and still be retained unchanged after targeted validation.

### 4.2 Initial Validation Candidates

The profiling results and the characteristics of the selected OSM
classifications are used to construct an initial set of mappings requiring
targeted validation.

Candidate mappings are selected when the profiling results reveal a
characteristic that may affect their interpretation or practical use in
ColMaps. The reason for selecting each mapping is recorded explicitly so that
the subsequent validation decisions remain traceable.

This initial selection does not modify the candidate feature scope. Its purpose
is only to determine which mappings require additional investigation and what
questions should be addressed during that investigation.


In [21]:
## GeneraL Structure to add potential candidates for revision
validation_candidates = []

## Helper to add candidates
def add_validation_candidate(
    osm_key,
    osm_value,
    risk_type,
    validation_question,
):
    validation_candidates.append({
        "osm_key": osm_key,
        "osm_value": osm_value,
        "risk_type": risk_type,
        "validation_question": validation_question,
    })

In [22]:
## Table merging the 3 signals previously get in the step 3 with descending sort
feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "named_percentage",
        "point_count",
        "linestring_count",
        "multilinestring_count",
        "polygon_count",
        "multipolygon_count",
        "geometry_types",
    ]
].sort_values(
    "object_count",
    ascending=False,
).head(30)

,category,osm_key,osm_value,object_count,named_percentage,point_count,linestring_count,multilinestring_count,polygon_count,multipolygon_count,geometry_types
26,nature,natural,water,42569,6.9,54,10,52,41870,112,5
59,parks_recreation,leisure,park,14898,40.4,1007,0,0,13776,69,3
68,food_drink,amenity,restaurant,14815,88.4,12073,0,3,2736,3,4
111,transport_travel,amenity,parking,10130,9.9,1310,0,1,8799,14,4
65,parks_recreation,leisure,swimming_pool,9553,5.0,160,0,3,9370,10,4
27,nature,natural,wetland,8208,4.2,146,1,1,7009,30,5
28,nature,natural,peak,7159,93.4,7159,0,0,0,0,1
93,markets_shopping,shop,convenience,6955,83.1,6073,0,0,880,2,3
74,accommodation,tourism,hotel,5655,91.2,4195,0,0,1453,4,3
94,markets_shopping,shop,supermarket,5601,90.4,3935,0,0,1659,2,3


In [23]:
## Clearly Candidates 

add_validation_candidate(
    "natural",
    "water",
    "semantic_breadth",
    "Does natural=water represent traveler-relevant destinations directly, "
    "or should secondary tags be used to distinguish relevant water features?",
)

add_validation_candidate(
    "leisure",
    "swimming_pool",
    "secondary_tag_dependency",
    "Do swimming pools represent traveler-relevant recreational destinations, "
    "or does the mapping include private, residential, or otherwise unsuitable facilities?",
)

add_validation_candidate(
    "natural",
    "wetland",
    "semantic_breadth",
    "Are wetlands generally meaningful route-oriented destinations, or is "
    "additional classification required to identify traveler-relevant features?",
)

add_validation_candidate(
    "leisure",
    "garden",
    "secondary_tag_dependency",
    "Does leisure=garden identify visitor-oriented destinations, or does it "
    "also include private or otherwise non-visitor-oriented gardens?",
)

add_validation_candidate(
    "natural",
    "cliff",
    "spatial_representation",
    "Can predominantly linear cliff geometries be meaningfully represented as "
    "route-oriented destinations in ColMaps?",
)

add_validation_candidate(
    "natural",
    "bare_rock",
    "spatial_representation",
    "Do predominantly polygonal bare-rock features represent meaningful "
    "traveler destinations, or are they primarily descriptive geographic areas?",
)

add_validation_candidate(
    "amenity",
    "parking_entrance",
    "semantic_relevance",
    "Should parking entrances be treated as traveler destinations, or should "
    "ColMaps represent the associated parking facilities instead?",
)

add_validation_candidate(
    "tourism",
    "artwork",
    "semantic_breadth",
    "Does tourism=artwork consistently represent visitor-relevant cultural "
    "destinations, or does its broad classification require refinement?",
)

In [24]:
validation_candidates_df = pd.DataFrame(validation_candidates)

validation_candidates_df

,osm_key,osm_value,risk_type,validation_question
0,natural,water,semantic_breadth,Does natural=water represent traveler-relevant...
1,leisure,swimming_pool,secondary_tag_dependency,Do swimming pools represent traveler-relevant ...
2,natural,wetland,semantic_breadth,Are wetlands generally meaningful route-orient...
3,leisure,garden,secondary_tag_dependency,Does leisure=garden identify visitor-oriented ...
4,natural,cliff,spatial_representation,Can predominantly linear cliff geometries be m...
5,natural,bare_rock,spatial_representation,Do predominantly polygonal bare-rock features ...
6,amenity,parking_entrance,semantic_relevance,Should parking entrances be treated as travele...
7,tourism,artwork,semantic_breadth,Does tourism=artwork consistently represent vi...


In [25]:
## Searching For more candidates with low disponibility of name
low_name_mappings = feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "named_percentage",
    ]
].sort_values(
    ["named_percentage", "object_count"],
    ascending=[True, False],
).head(25)

low_name_mappings

,category,osm_key,osm_value,object_count,named_percentage
61,parks_recreation,leisure,picnic_table,116,0.0
41,nature,natural,desert,63,0.0
25,history_heritage,man_made,ceremonial_gate,3,0.0
52,nature,natural,islet,2,0.0
55,nature,natural,peninsula,1,0.0
112,transport_travel,amenity,parking_entrance,1045,2.1
29,nature,natural,cliff,3674,3.3
60,parks_recreation,leisure,garden,4805,3.7
27,nature,natural,wetland,8208,4.2
65,parks_recreation,leisure,swimming_pool,9553,5.0


In [26]:
## Predominantly linear mappings
feature_profile["linear_count"] = (
    feature_profile["linestring_count"]
    + feature_profile["multilinestring_count"]
)

feature_profile["linear_percentage"] = (
    feature_profile["linear_count"]
    / feature_profile["object_count"]
    * 100
).round(1)

linear_mappings = feature_profile[
    [
        "category",
        "osm_key",
        "osm_value",
        "object_count",
        "linear_count",
        "linear_percentage",
    ]
].sort_values(
    ["linear_percentage", "object_count"],
    ascending=[False, False],
).head(20)

linear_mappings

,category,osm_key,osm_value,object_count,linear_count,linear_percentage
32,nature,natural,ridge,540,538,99.6
29,nature,natural,cliff,3674,3527,96.0
45,nature,natural,valley,22,12,54.5
46,nature,natural,mountain_range,12,1,8.3
6,museums_culture,tourism,artwork,1022,21,2.1
35,nature,natural,reef,156,3,1.9
57,nature,waterway,waterfall,772,12,1.6
112,transport_travel,amenity,parking_entrance,1045,14,1.3
0,attractions,tourism,attraction,1194,13,1.1
38,nature,natural,bay,117,1,0.9


In [27]:
## Risk found
add_validation_candidate(
    "natural",
    "ridge",
    "spatial_representation",
    "Can predominantly linear ridge geometries be meaningfully represented as "
    "route-oriented destinations in ColMaps?",
)

In [28]:
## This case is very specific because the tag is the same, but they belong to different categories.
add_validation_candidate(
    "natural",
    "waterfall",
    "overlapping_classification",
    "How does natural=waterfall differ from waterway=waterfall in the Colombia "
    "dataset, and should both tagging conventions be retained?",
)

add_validation_candidate(
    "waterway",
    "waterfall",
    "overlapping_classification",
    "How does waterway=waterfall differ from natural=waterfall in the Colombia "
    "dataset, and should both tagging conventions be retained?",
)

In [29]:
## Deleting Duplicates if exists
validation_candidates_df = (
    pd.DataFrame(validation_candidates)
    .drop_duplicates(
        subset=["osm_key", "osm_value", "risk_type"]
    )
    .reset_index(drop=True)
)

validation_candidates_df

,osm_key,osm_value,risk_type,validation_question
0,natural,water,semantic_breadth,Does natural=water represent traveler-relevant...
1,leisure,swimming_pool,secondary_tag_dependency,Do swimming pools represent traveler-relevant ...
2,natural,wetland,semantic_breadth,Are wetlands generally meaningful route-orient...
3,leisure,garden,secondary_tag_dependency,Does leisure=garden identify visitor-oriented ...
4,natural,cliff,spatial_representation,Can predominantly linear cliff geometries be m...
5,natural,bare_rock,spatial_representation,Do predominantly polygonal bare-rock features ...
6,amenity,parking_entrance,semantic_relevance,Should parking entrances be treated as travele...
7,tourism,artwork,semantic_breadth,Does tourism=artwork consistently represent vi...
8,natural,ridge,spatial_representation,Can predominantly linear ridge geometries be m...
9,natural,waterfall,overlapping_classification,How does natural=waterfall differ from waterwa...


#### Initial validation candidate observations

The risk-identification process selected 11 candidate mappings for additional
investigation. These mappings were not selected through a single quantitative
threshold; instead, they were identified by combining the profiling results
with the semantic role of each feature within ColMaps.

Several natural-feature mappings, including `natural=water` and
`natural=wetland`, were selected because their broad classifications and low
name availability may encompass geographic objects with substantially different
relevance to travelers. `leisure=swimming_pool` and `leisure=garden` similarly
require investigation because their primary classifications may not distinguish
visitor-oriented facilities from private or otherwise unsuitable features.

Spatial representation identified additional cases. `natural=cliff` and
`natural=ridge` are represented predominantly through linear geometries, while
`natural=bare_rock` primarily represents geographic areas. Their suitability
therefore depends not only on their semantic classification but also on how such
features can be represented meaningfully as route-oriented destinations.

`amenity=parking_entrance` was selected because an entrance may represent an
access point to a parking facility rather than a traveler service that should
be presented independently. `tourism=artwork` was selected due to the breadth
of the classification and the variety of real-world objects that it may
represent.

Finally, `natural=waterfall` and `waterway=waterfall` were selected together
because they represent overlapping OSM tagging conventions for the same general
type of natural attraction. Their observed use in the dataset requires
comparison before determining the final mapping strategy.

These mappings form the initial targeted-validation set. Their inclusion in
this set does not imply that they will be removed or modified; each case must
first be investigated using the underlying OSM attributes and observed dataset
characteristics.

## 5. Targeted Validation

The mappings identified during risk identification are examined individually
using the underlying OSM data.

Unlike the general profiling stage, each investigation is driven by a specific
validation question. Secondary OSM tags, feature characteristics, and observed
mapping patterns are examined only when they are relevant to resolving that
question.

The objective is to determine whether each investigated mapping can be retained
unchanged, requires refinement through additional filtering rules, or should be
excluded from the final ColMaps feature scope.

Each validation case records the observed evidence and the resulting decision
so that modifications to the candidate scope remain explicit and reproducible.

### 5.1 natural=water

Question:

`Does natural=water represent traveler-relevant destinations directly, or should secondary tags be used to distinguish relevant water features?`

Profiling:
- 42,466 objects
- 7.0% named
- mostly Polygon

The natural=water mapping contains 42,466 objects, making it the most frequent
individual mapping in the candidate scope. Only approximately 7% of these
objects provide an explicit name, and the mapping is represented
predominantly through polygon geometries.

However, natural=water describes a broad class of water features and does not
by itself indicate the specific type of water body represented by each object.
The mapping is therefore investigated using secondary OSM attributes to
determine whether additional classification is required for ColMaps.

In [30]:
# Temporary extract used only for the natural=water validation case.
WATER_VALIDATION_PBF = Path("../filtered/validation_natural_water.osm.pbf")

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "natural=water",
        "-o",
        str(WATER_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'natural=water', '-o', '..\\filtered\\validation_natural_water.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [31]:
# We count water=* within that subset

result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(WATER_VALIDATION_PBF),
        "water=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

water_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    water_rows.append({
        "water": value.strip('"'),
        "object_count": int(count),
    })

water_secondary_tags = pd.DataFrame(water_rows)

In [32]:
print("Water tags Count: ", len(water_secondary_tags))
water_secondary_tags.head(50)

Water tags Count:  44


,water,object_count
0,pond,11852
1,river,3786
2,lake,2788
3,reservoir,2017
4,basin,221
5,oxbow,205
6,canal,118
7,wastewater,108
8,fishpond,74
9,lagoon,57


#### Validation result

The secondary-tag analysis identified 44 distinct `water=*` values associated
with the investigated `natural=water` population. The distribution is dominated
by several established classifications, particularly `pond`, `river`, `lake`,
and `reservoir`, while smaller groups include features such as `basin`,
`oxbow`, `canal`, `wastewater`, `fishpond`, and `lagoon`.

The observed values demonstrate that `natural=water` represents a heterogeneous
collection of water features rather than a single type of traveler-oriented
destination. Some classifications may represent potentially relevant natural
places, while others describe infrastructure, artificial water features, or
objects whose relevance depends strongly on additional context.

The secondary tag also contains a small number of non-standard or free-text
values, indicating that `water=*` cannot be treated as an unrestricted
classification field.

Furthermore, not every `natural=water` object is represented by one of the
observed secondary `water=*` classifications. Consequently,
`natural=water` alone does not provide sufficient semantic specificity for
direct inclusion as a uniform ColMaps feature type.

**Decision: REFINE.** The `natural=water` mapping is retained as a candidate
source classification, but the final dataset-preparation rules should use
validated secondary attributes to distinguish water features suitable for
ColMaps from overly broad, infrastructural, or otherwise unsuitable objects.

### 5.2 leisure=swimming_pool

Question:

`Does leisure=swimming_pool identify recreational facilities useful to travelers, or are we including private/residential pools and the like?`

Profiling: 
- 9,500 objects
- 5.0% named
- Mostly Polygon

The leisure=swimming_pool mapping contains 9,500 objects and is represented
almost entirely through polygon geometries. Only approximately 5% of the
objects provide an explicit name.

Low name availability is not inherently problematic for swimming pools, since
individual pools may legitimately exist without a distinct name. The primary
validation concern is instead whether the mapping distinguishes
visitor-oriented recreational facilities from private, residential, or other
restricted swimming pools.

Secondary OSM attributes are therefore examined to determine whether access or
related contextual information can support a more precise ColMaps
classification.


In [33]:
# Temporary extract used only for the leisure=swimming_pool validation case.
SWIMMING_POOL_VALIDATION_PBF = Path(
    "../filtered/validation_leisure_swimming_pool.osm.pbf"
)

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "leisure=swimming_pool",
        "-o",
        str(SWIMMING_POOL_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'leisure=swimming_pool', '-o', '..\\filtered\\validation_leisure_swimming_pool.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [34]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(SWIMMING_POOL_VALIDATION_PBF),
        "access=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

access_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    access_rows.append({
        "access": value.strip('"'),
        "object_count": int(count),
    })

swimming_pool_access = pd.DataFrame(access_rows)


In [35]:
print(f"Access tags count: {len(swimming_pool_access):,}")
swimming_pool_access

Access tags count: 7


,access,object_count
0,private,900
1,customers,236
2,yes,101
3,permissive,49
4,no,8
5,permit,8
6,unknown,2


In [36]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(SWIMMING_POOL_VALIDATION_PBF),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

swimming_pool_tag_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 2:
        continue

    count, key = parts

    swimming_pool_tag_rows.append({
        "tag": key.strip('"'),
        "object_count": int(count),
    })

swimming_pool_tags = pd.DataFrame(swimming_pool_tag_rows)

In [37]:
print('Swimming pool tags', len(swimming_pool_tags))
swimming_pool_tags.head(20)

Swimming pool tags 72


,tag,object_count
0,leisure,9556
1,access,1304
2,name,483
3,location,445
4,sport,359
5,swimming_pool,239
6,source,185
7,lit,115
8,addr:city,89
9,covered,60


#### Validation result

The access analysis identified explicit `access=*` information for 1,298
objects. Among these, 897 are classified as `private`, while smaller groups are
classified as `customers`, `yes`, `permissive`, `no`, or `permit`. This confirms
that `leisure=swimming_pool` includes facilities with substantially different
access conditions and cannot be interpreted uniformly as a publicly accessible
recreational destination.

However, `access=*` is available for only a minority of the approximately 9,500
candidate swimming pools. Inspection of the remaining secondary tag coverage
shows that no alternative contextual attribute is sufficiently widespread to
provide a complete classification. Tags such as `location`, `sport`, and
`swimming_pool` occur considerably less frequently, while explicit
`opening_hours` and `fee` information is rare.

Consequently, the absence of an `access` value cannot be interpreted as
evidence of public accessibility, but it also cannot be used to exclude a
swimming pool automatically.

**Decision: REFINE.** The `leisure=swimming_pool` mapping is retained, but
explicit access restrictions should be considered during final dataset
preparation. In particular, objects explicitly identified as inaccessible or
private should not be treated in the same manner as publicly accessible
recreational facilities. Objects without access information require
conservative handling rather than an assumed accessibility status.

### 5.3 natural=wetland

Question:

`Are wetlands generally meaningful route-oriented destinations, or is additional classification required to identify traveler-relevant features?`

Profiling:

* 8,185 objects
* 4.2% named
* Mostly Polygon

The `natural=wetland` mapping contains 8,185 objects and is represented
predominantly through polygon geometries. Only approximately 4.2% of the
objects provide an explicit name.

Low name availability is not inherently problematic for wetlands, since natural
geographic features may legitimately exist without individual names. The
primary validation concern is instead whether the broad `natural=wetland`
classification provides sufficient information to determine their relevance as
places of interest for travelers.

The secondary `wetland=*` classification is therefore examined to determine
which types of wetlands occur within the candidate population and whether this
additional information can support a more precise ColMaps classification.



In [38]:
# Temporary extract used only for the natural=wetland validation case.
WETLAND_VALIDATION_PBF = Path(
    "../filtered/validation_natural_wetland.osm.pbf"
)

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "natural=wetland",
        "-o",
        str(WETLAND_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'natural=wetland', '-o', '..\\filtered\\validation_natural_wetland.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [39]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(WETLAND_VALIDATION_PBF),
        "wetland=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

wetland_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    wetland_rows.append({
        "wetland": value.strip('"'),
        "object_count": int(count),
    })

wetland_secondary_tags = pd.DataFrame(wetland_rows)

In [40]:
print(f"Wetland tags count: {len(wetland_secondary_tags):,}")

wetland_secondary_tags

Wetland tags count: 39


,wetland,object_count
0,mangrove,2103
1,wet_meadow,546
2,tidalflat,441
3,reedbed,383
4,swamp,326
5,marsh,144
6,bog,111
7,saltmarsh,20
8,saltern,16
9,arroyo,4


#### Validation result

The secondary-tag analysis identified 39 distinct `wetland=*` values within
the investigated `natural=wetland` population. The most frequent established
classifications include `mangrove`, `wet_meadow`, `tidalflat`, `reedbed`,
`swamp`, `marsh`, and `bog`.

The observed distribution confirms that `natural=wetland` represents a
heterogeneous collection of natural environments rather than a single type of
traveler-oriented destination. The secondary classification provides
substantially more semantic information about the physical characteristics of
individual wetlands.

However, `wetland=*` is available for only approximately half of the observed
`natural=wetland` population. In addition, a small number of values contain
non-standard classifications or free-text descriptions, demonstrating that the
secondary tag cannot be used as an unrestricted classification field.

Consequently, `natural=wetland` alone does not provide sufficient semantic
specificity to determine whether every mapped wetland should be presented as a
ColMaps destination, while the secondary `wetland=*` tag provides useful but
incomplete contextual information.

**Decision: REFINE.** The `natural=wetland` mapping is retained as a candidate
source classification, but the final dataset-preparation rules should consider
validated secondary attributes and additional contextual information when
determining which wetland features are suitable for ColMaps.


### 5.4 leisure=garden

Question:

`Does leisure=garden identify visitor-oriented destinations, or does it also include private or otherwise non-visitor-oriented gardens?`

Profiling:

* 4,798 objects
* 3.7% named
* Mostly Polygon

The `leisure=garden` mapping contains 4,798 objects and is represented predominantly through polygon geometries. Only approximately 3.7% of the objects provide an explicit name.

Low name availability is not necessarily problematic for gardens, since mapped garden areas may legitimately exist without individual names. The primary validation concern is instead whether `leisure=garden` consistently represents places that can be meaningfully presented to travelers.

As with swimming pools, access conditions may provide useful evidence for distinguishing publicly accessible or visitor-oriented gardens from private or restricted spaces. Secondary OSM attributes are therefore examined to determine whether the primary classification requires additional refinement.




In [41]:
# Temporary extract used only for the leisure=garden validation case.
GARDEN_VALIDATION_PBF = Path(
    "../filtered/validation_leisure_garden.osm.pbf"
)

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "leisure=garden",
        "-o",
        str(GARDEN_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'leisure=garden', '-o', '..\\filtered\\validation_leisure_garden.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [42]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(GARDEN_VALIDATION_PBF),
        "access=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

garden_access_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    garden_access_rows.append({
        "access": value.strip('"'),
        "object_count": int(count),
    })

garden_access = pd.DataFrame(garden_access_rows)

In [43]:
print(f"Access tags count: {len(garden_access):,}")

garden_access

Access tags count: 6


,access,object_count
0,yes,54
1,private,40
2,permissive,17
3,customers,15
4,no,4
5,permit,2


In [44]:
## Only Few access less than 2%... Better to check what is inside of the whole tag
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(GARDEN_VALIDATION_PBF),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

garden_tag_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 2:
        continue

    count, key = parts

    garden_tag_rows.append({
        "tag": key.strip('"'),
        "object_count": int(count),
    })

garden_tags = pd.DataFrame(garden_tag_rows)


In [45]:
print(f"Garden tags count: {len(garden_tags):,}")

garden_tags.head(30)

Garden tags count: 113


,tag,object_count
0,leisure,4805
1,garden:type,311
2,garden:style,278
3,name,223
4,access,132
5,highway,70
6,description,69
7,landuse,67
8,note,49
9,height,47


#### Validation result

The access analysis identified explicit `access=*` information for only 132 of
the 4,798 observed gardens. These values include publicly accessible or
permissive cases as well as `private`, `customers`, and `no`, confirming that
`leisure=garden` does not represent a uniformly accessible population.

The broader secondary-tag inspection identified 113 different tag keys, but
none provides sufficient coverage to classify the complete population.
`garden:type` and `garden:style` are the most frequent contextual attributes,
but occur for only a small proportion of the observed objects. Explicit
information such as `access`, `operator`, and `fee` is even less frequent.

Consequently, secondary attributes can provide useful evidence for individual
objects but cannot reliably determine visitor suitability across the complete
`leisure=garden` population. The absence of access information must not be
interpreted as evidence of either public or private accessibility.

**Decision: REFINE.** The `leisure=garden` mapping is retained as a candidate
source classification, but explicit access restrictions and other available
contextual attributes should be considered during final dataset preparation.
Objects without such information require conservative handling rather than an
assumed visitor-access status.

### 5.5 natural=cliff

Question:

`Can predominantly linear cliff geometries be meaningfully represented as route-oriented destinations in ColMaps?`

Profiling:

* 3,672 objects
* 3.3% named
* 96.0% linear geometries

The `natural=cliff` mapping contains 3,672 objects and differs structurally from
most conventional destination-oriented features in the candidate scope.
Approximately 96% of the observed objects are represented through linear
geometries, primarily `MultiLineString`.

This representation is consistent with the physical nature of cliffs, which
may extend across a geographic boundary rather than correspond to a single
point location. Consequently, the validation concern is not whether the OSM
classification itself is semantically meaningful, but whether its spatial
representation can be integrated into the route-oriented ColMaps model.

The geometry characteristics are therefore examined to determine whether linear
cliff features can be transformed into a representative location for proximity
and route-based processing without discarding their original spatial geometry.


    

In [46]:
# Select the natural=cliff candidate objects.
cliff_features = candidate_features[
    candidate_features["natural"] == "cliff"
].copy()

# Reproject to a metric CRS before calculating geometry length.
cliff_features_metric = cliff_features.to_crs(
    cliff_features.estimate_utm_crs()
)

# Calculate the length of each reconstructed geometry in meters.
cliff_features_metric["length_m"] = (
    cliff_features_metric.geometry.length
)

cliff_features_metric["length_m"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count     3632.000000
mean       697.728196
std       1587.660630
min          0.000000
25%        145.182741
50%        322.199294
75%        708.072336
90%       1483.540487
95%       2407.677157
99%       6413.250561
max      53799.440578
Name: length_m, dtype: float64

#### Validation result

The geometry analysis confirms that `natural=cliff` is predominantly represented through linear geometries, but the observed spatial extent of most features remains relatively localized.

The median geometry length is approximately 322 m, while 75% of the measured features are shorter than approximately 708 m and 90% are shorter than approximately 1.48 km. Larger geometries occur less frequently, although a small number of substantial outliers are present, with the largest observed geometry extending approximately 53.8 km.

These results indicate that linear representation does not prevent cliffs from being used as route-oriented geographic features. Their original geometry can be preserved for spatial operations such as proximity analysis, while a representative location may be derived later when a point-based representation is required for presentation or other application functionality.

The presence of unusually large geometries should be considered during later spatial processing, but it does not indicate that the underlying `natural=cliff` classification requires semantic refinement.

**Decision: RETAIN.** The `natural=cliff` mapping is retained unchanged in the validated feature scope. Its linear geometry is treated as a spatial-processing consideration rather than a feature-classification problem.

### 5.6 natural=bare_rock

Question:

`Do predominantly polygonal bare-rock features represent meaningful traveler destinations, or are they primarily descriptive geographic areas?`

Profiling:

* 2,654 objects
* 5.2% named
* Mostly Polygon

The `natural=bare_rock` mapping contains 2,654 objects and is represented
predominantly through polygon geometries. Only approximately 5.2% of the
observed objects provide an explicit name.

Unlike conventional point-based destinations, `natural=bare_rock` describes
areas where exposed bedrock forms the dominant surface. The primary validation
concern is therefore whether the observed geometries represent reasonably
localized natural features that can participate meaningfully in route-oriented
discovery, or whether the mapping primarily consists of extensive descriptive
geographic areas.

The spatial extent of the observed geometries is therefore examined before
determining whether the mapping requires refinement or exclusion.

In [47]:
# Select the natural=bare_rock candidate objects.
bare_rock_features = candidate_features[
    candidate_features["natural"] == "bare_rock"
].copy()

# Reproject to a metric CRS before calculating geometry area.
bare_rock_features_metric = bare_rock_features.to_crs(
    bare_rock_features.estimate_utm_crs()
)

# Calculate the area of each reconstructed geometry in square meters.
bare_rock_features_metric["area_m2"] = (
    bare_rock_features_metric.geometry.area
)

bare_rock_features_metric["area_m2"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    2.457000e+03
mean     1.109002e+06
std      6.372010e+06
min      0.000000e+00
25%      5.163707e+03
50%      3.455156e+04
75%      2.574207e+05
90%      1.257014e+06
95%      3.363625e+06
99%      2.349880e+07
max      1.188021e+08
Name: area_m2, dtype: float64

In [48]:
# Convert square meters to square kilometers for easier interpretation.
bare_rock_features_metric["area_km2"] = (
    bare_rock_features_metric["area_m2"] / 1_000_000
)

bare_rock_features_metric["area_km2"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    2457.000000
mean        1.109002
std         6.372010
min         0.000000
25%         0.005164
50%         0.034552
75%         0.257421
90%         1.257014
95%         3.363625
99%        23.498805
max       118.802126
Name: area_km2, dtype: float64

#### Validation result

The area analysis shows that most `natural=bare_rock` geometries represent relatively localized geographic features, despite the presence of a smaller number of substantially larger areas.

The median observed area is approximately 0.035 km², while 75% of the measured features are smaller than approximately 0.261 km² and 90% are smaller than approximately 1.26 km². Larger geographic areas occur progressively less frequently, although substantial outliers are present, with the largest observed geometry covering approximately 118.8 km².

The strongly skewed distribution indicates that the high maximum and mean values are influenced by a relatively small population of large geometries rather than representing the typical `natural=bare_rock` feature.

Consequently, polygon representation does not by itself prevent these features from participating in route-oriented spatial analysis. Their original geometry can be preserved for proximity operations, while representative locations may be derived later when required by point-based application functionality.

**Decision: RETAIN.** The `natural=bare_rock` mapping is retained unchanged in the validated feature scope. Large-area outliers are treated as a later spatial-processing consideration rather than as evidence that the underlying OSM classification requires refinement.

### 5.7 amenity=parking_entrance

Question:

`Should parking entrances be treated as traveler destinations, or should ColMaps represent the associated parking facilities instead?`

Profiling:

* 1,044 objects
* 2.1% named
* Mostly Point

The `amenity=parking_entrance` mapping contains 1,044 objects and is represented almost entirely through point geometries. Only approximately 2.1% of the observed objects provide an explicit name.

Unlike the previous spatial-representation cases, the primary concern is not whether these objects can be located or processed spatially. A parking entrance already provides a precise geographic location. The question is instead whether the entrance itself represents the traveler service that ColMaps intends to present.

ColMaps includes parking as a traveler service associated with a route. A parking entrance may represent an access point to such a facility rather than an independent destination. Secondary OSM attributes are therefore examined to determine whether `amenity=parking_entrance` provides meaningful information as a standalone feature or primarily describes access infrastructure associated with parking facilities.


In [49]:
# Temporary extract used only for the amenity=parking_entrance validation case.
PARKING_ENTRANCE_VALIDATION_PBF = Path(
    "../filtered/validation_amenity_parking_entrance.osm.pbf"
)

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "amenity=parking_entrance",
        "-o",
        str(PARKING_ENTRANCE_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'amenity=parking_entrance', '-o', '..\\filtered\\validation_amenity_parking_entrance.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [50]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(PARKING_ENTRANCE_VALIDATION_PBF),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

parking_entrance_tag_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 2:
        continue

    count, key = parts

    parking_entrance_tag_rows.append({
        "tag": key.strip('"'),
        "object_count": int(count),
    })

parking_entrance_tags = pd.DataFrame(
    parking_entrance_tag_rows
)

In [51]:
print(
    f"Parking entrance tags count: "
    f"{len(parking_entrance_tags):,}"
)

parking_entrance_tags.head(30)

Parking entrance tags count: 48


,tag,object_count
0,amenity,1045
1,parking,190
2,access,180
3,barrier,72
4,fee,30
5,name,22
6,supervised,19
7,entrance,18
8,highway,15
9,surface,15


#### Validation result

The secondary-tag analysis identified 48 different tag keys associated with `amenity=parking_entrance`. The most frequent contextual attributes include `parking`, `access`, `barrier`, and `fee`, while descriptive attributes such as `name`, `operator`, and `opening_hours` occur considerably less frequently.

The observed attributes primarily describe access conditions and physical characteristics of the entrance rather than an independent traveler service. Tags such as `access`, `barrier`, `maxheight`, `oneway`, and vehicle-related restrictions are consistent with the role of these objects as access points to parking infrastructure.

ColMaps already represents parking facilities through the `amenity=parking` mapping. From the route-oriented application perspective, the parking facility represents the traveler service of interest, while a `parking_entrance` primarily describes how that facility can be accessed.

**Decision: EXCLUDE.** The `amenity=parking_entrance` mapping is excluded from the final ColMaps feature scope as an independent traveler destination or service. Parking entrances may remain relevant to later routing or access processing, but they should not be presented as standalone ColMaps features.


### 5.8 tourism=artwork

Question:

`Does tourism=artwork consistently represent visitor-relevant cultural destinations, or does its broad classification require refinement?`

Profiling:

* 1,021 objects
* 5 geometry types
* Mostly Point

The `tourism=artwork` mapping contains 1,021 objects and is represented predominantly through point geometries, although all five geometry types observed in the candidate dataset are present.

Unlike the previous spatial-representation cases, geometry diversity does not represent the primary validation concern. The `tourism=artwork` classification can describe multiple forms of public or visitor-oriented artwork, potentially including substantially different types of cultural objects.

The secondary `artwork_type=*` classification is therefore examined to determine which forms of artwork occur within the candidate population and whether the general `tourism=artwork` mapping provides sufficient semantic specificity for ColMaps.


In [52]:
# Temporary extract used only for the tourism=artwork validation case.
ARTWORK_VALIDATION_PBF = Path(
    "../filtered/validation_tourism_artwork.osm.pbf"
)

subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(CANDIDATE_PBF_PATH),
        "tourism=artwork",
        "-o",
        str(ARTWORK_VALIDATION_PBF),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

CompletedProcess(args=['osmium', 'tags-filter', '..\\filtered\\01_candidate_scope.osm.pbf', 'tourism=artwork', '-o', '..\\filtered\\validation_tourism_artwork.osm.pbf', '--overwrite'], returncode=0, stdout='', stderr='')

In [53]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(ARTWORK_VALIDATION_PBF),
        "artwork_type=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

artwork_type_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    artwork_type_rows.append({
        "artwork_type": value.strip('"'),
        "object_count": int(count),
    })

artwork_types = pd.DataFrame(artwork_type_rows)


In [54]:

print(f"Artwork types count: {len(artwork_types):,}")

artwork_types

Artwork types count: 19


,artwork_type,object_count
0,statue,285
1,sculpture,239
2,mural,177
3,bust,82
4,graffiti,31
5,installation,9
6,architecture,5
7,stone,5
8,fountain,3
9,mosaic,3


#### Validation result

The secondary-tag analysis identified 19 distinct `artwork_type=*`classifications. Approximately 83% of the observed `tourism=artwork` objects provide this secondary classification, representing substantially higher coverage than the contextual attributes examined for several previous
validation cases.

The distribution is dominated by established artwork types. `statue`, `sculpture`, `mural`, `bust`, and `graffiti` account for the majority of the classified population, while additional values such as `installation`, `mosaic`, `painting`, `tilework`, and `relief` represent less frequent but semantically consistent forms of artwork.

A small number of non-standard or free-text values are present, but these occur only as isolated cases and do not characterize the overall mapping population.

The observed secondary classifications therefore support the interpretation of `tourism=artwork` as a coherent general category of cultural features. The variation between individual artwork types provides useful descriptive information but does not require separate inclusion rules for the current ColMaps feature scope.

**Decision: RETAIN.** The `tourism=artwork` mapping is retained unchanged in the validated feature scope. `artwork_type=*` may be preserved as descriptive metadata where available, but it is not required as an inclusion condition.

### 5.9 natural=ridge

Question:

`Can predominantly linear ridge geometries be meaningfully represented as route-oriented destinations in ColMaps?`

Profiling:

* 540 objects
* 11.1% named
* 99.6% linear geometries

The `natural=ridge` mapping contains 540 objects and exhibits the strongest
linear representation among the candidate mappings. Approximately 99.6% of the
observed objects are represented through linear geometries.

This representation is consistent with the physical nature of a ridge, which
normally extends along an elevated geographic structure rather than
corresponding to a single geographic point. Similar to `natural=cliff`, the
primary validation concern is therefore spatial rather than semantic.

The spatial extent of the observed ridge geometries is examined to determine
whether they can participate meaningfully in route-oriented proximity analysis
while preserving their original linear representation.


In [55]:
# Select the natural=ridge candidate objects.
ridge_features = candidate_features[
    candidate_features["natural"] == "ridge"
].copy()

# Reproject to a metric CRS before calculating geometry length.
ridge_features_metric = ridge_features.to_crs(
    ridge_features.estimate_utm_crs()
)

# Calculate the length of each reconstructed geometry in meters.
ridge_features_metric["length_m"] = (
    ridge_features_metric.geometry.length
)

ridge_features_metric["length_m"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count      540.000000
mean      1223.261976
std       3326.102023
min          4.470608
25%        129.436965
50%        317.194467
75%        995.975541
90%       2656.379705
95%       4931.783326
99%      14581.349438
max      45724.893392
Name: length_m, dtype: float64

#### Validation result

The geometry analysis confirms that `natural=ridge` is almost exclusively
represented through linear geometries, but the spatial extent of most observed
features remains relatively localized.

The median geometry length is approximately 317 m, while 75% of the measured
features are shorter than approximately 996 m and 90% are shorter than
approximately 2.66 km. Longer ridge geometries occur progressively less
frequently, although substantial outliers are present, with the largest
observed geometry extending approximately 45.7 km.

The distribution therefore shows that linear representation is an expected
structural characteristic of ridges rather than evidence of an unsuitable OSM
classification. The original geometry can be preserved for spatial operations
such as route proximity analysis, while a representative location may be
derived later when point-based application functionality requires one.

**Decision: RETAIN.** The `natural=ridge` mapping is retained unchanged in the
validated feature scope. Its predominantly linear representation and occasional
large spatial extent are treated as later spatial-processing considerations
rather than feature-classification problems.

### 5.10 Waterfall Tagging Conventions

Question:

`How do natural=waterfall and waterway=waterfall differ in the Colombia dataset, and should both tagging conventions be retained?`

Profiling:

* `natural=waterfall`: 5 objects
* `waterway=waterfall`: 772 objects
* Both mappings represent the same general type of natural feature

Waterfalls appear in the candidate scope through two different OSM tagging
conventions: `natural=waterfall` and `waterway=waterfall`. Their observed
frequencies differ substantially, with only 5 objects using the former
classification compared with 772 using the latter.

The validation concern is therefore not the suitability of waterfalls as
traveler-oriented natural destinations, but whether these two mappings
represent distinct populations or overlapping tagging conventions for the same
real-world feature type.

The two populations are examined together to determine whether both mappings
should be retained, consolidated, or otherwise handled during final dataset
preparation.



In [56]:
# Select objects using either of the two waterfall tagging conventions.
natural_waterfalls = candidate_features[
    candidate_features["natural"] == "waterfall"
].copy()

waterway_waterfalls = candidate_features[
    candidate_features["waterway"] == "waterfall"
].copy()

# Identify objects where both classifications occur on the same OSM feature.
combined_waterfalls = candidate_features[
    (candidate_features["natural"] == "waterfall")
    & (candidate_features["waterway"] == "waterfall")
].copy()

print(f"natural=waterfall: {len(natural_waterfalls):,}")
print(f"waterway=waterfall: {len(waterway_waterfalls):,}")
print(f"Both tags on the same object: {len(combined_waterfalls):,}")

natural=waterfall: 5
waterway=waterfall: 772
Both tags on the same object: 2


In [57]:
# Reproject both waterfall populations to a common metric CRS.
waterfall_crs = waterway_waterfalls.estimate_utm_crs()

natural_waterfalls_metric = natural_waterfalls.to_crs(waterfall_crs)
waterway_waterfalls_metric = waterway_waterfalls.to_crs(waterfall_crs)

distance_rows = []

# For each natural=waterfall object, find the nearest
# waterway=waterfall geometry.
for index, natural_feature in natural_waterfalls_metric.iterrows():

    distances = waterway_waterfalls_metric.geometry.distance(
        natural_feature.geometry
    )

    nearest_index = distances.idxmin()
    nearest_distance = distances.loc[nearest_index]

    distance_rows.append({
        "natural_osm_id": natural_feature["id"],
        "nearest_waterway_osm_id": (
            waterway_waterfalls_metric.loc[nearest_index, "id"]
        ),
        "distance_m": nearest_distance,
    })

waterfall_distances = pd.DataFrame(distance_rows)

waterfall_distances

,natural_osm_id,nearest_waterway_osm_id,distance_m
0,1023087149,1023087149,0.000000
1,1427108036,1427108036,0.000000
2,2077040383,6127302985,311.604746
3,4495512161,4718972689,9549.300371
4,5623291962,6275020955,15521.549820


#### Validation result

The comparison identified only limited direct overlap between the two waterfall
tagging conventions. Of the 5 objects classified as `natural=waterfall`, 2 also
use `waterway=waterfall` on the same OSM object.

A nearest-feature comparison was performed for the remaining population. One
`natural=waterfall` object is located approximately 312 m from the nearest
`waterway=waterfall`, while the remaining two are approximately 9.55 km and
15.52 km from their nearest corresponding features.

These results indicate that the two tagging conventions are not completely
interchangeable in the Colombia dataset. Although direct duplication exists,
removing `natural=waterfall` in favor of the substantially more frequent
`waterway=waterfall` convention would also remove waterfall objects that are not
represented by the latter classification.

The approximately 312 m case may represent either nearby distinct features or
different OSM representations of the same real-world waterfall. Determining
real-world equivalence from distance alone would require additional evidence
and is therefore not assumed during semantic scope validation.

**Decision: RETAIN BOTH.** Both `natural=waterfall` and
`waterway=waterfall` are retained in the validated feature scope. Objects
carrying both classifications should be consolidated as a single ColMaps
feature, while potential spatial duplicates represented by different OSM
objects may be addressed separately during later data-processing and
deduplication stages.


## 6. Validated Feature Scope

The targeted validation stage identified candidate mappings that can be
retained unchanged, mappings that require additional filtering conditions, and
one mapping that should not be represented as an independent ColMaps feature.

The purpose of this section is to consolidate these decisions into a validated
feature-scope specification that can be consumed by the subsequent dataset
preparation pipeline.

The validation decisions are represented using three states:

* **RETAIN** — the candidate `key=value` mapping remains part of the ColMaps
  feature scope without additional semantic filtering requirements.
* **REFINE** — the mapping remains relevant to ColMaps, but the primary
  `key=value` classification alone is insufficient and additional attributes or
  filtering rules must be considered.
* **EXCLUDE** — the mapping is removed from the final ColMaps feature scope as
  an independent destination or traveler service.

These decisions describe the semantic requirements of the dataset. Their
deterministic implementation against the original OSM source dataset is handled
by the subsequent data-preparation stage.

### 6.1 Validation Decisions

The targeted-validation results are first consolidated into a structured
decision table. Candidate mappings that were not selected for targeted
validation retain their original inclusion status, while investigated mappings
receive the decision established from the observed dataset evidence.


In [58]:
# Consolidate the decisions established during targeted validation.
validation_decisions = [
    {
        "osm_key": "natural",
        "osm_value": "water",
        "decision": "REFINE",
    },
    {
        "osm_key": "leisure",
        "osm_value": "swimming_pool",
        "decision": "REFINE",
    },
    {
        "osm_key": "natural",
        "osm_value": "wetland",
        "decision": "REFINE",
    },
    {
        "osm_key": "leisure",
        "osm_value": "garden",
        "decision": "REFINE",
    },
    {
        "osm_key": "natural",
        "osm_value": "cliff",
        "decision": "RETAIN",
    },
    {
        "osm_key": "natural",
        "osm_value": "bare_rock",
        "decision": "RETAIN",
    },
    {
        "osm_key": "amenity",
        "osm_value": "parking_entrance",
        "decision": "EXCLUDE",
    },
    {
        "osm_key": "tourism",
        "osm_value": "artwork",
        "decision": "RETAIN",
    },
    {
        "osm_key": "natural",
        "osm_value": "ridge",
        "decision": "RETAIN",
    },
    {
        "osm_key": "natural",
        "osm_value": "waterfall",
        "decision": "RETAIN",
    },
    {
        "osm_key": "waterway",
        "osm_value": "waterfall",
        "decision": "RETAIN",
    },
]

validation_decisions_df = pd.DataFrame(validation_decisions)

validation_decisions_df

,osm_key,osm_value,decision
0,natural,water,REFINE
1,leisure,swimming_pool,REFINE
2,natural,wetland,REFINE
3,leisure,garden,REFINE
4,natural,cliff,RETAIN
5,natural,bare_rock,RETAIN
6,amenity,parking_entrance,EXCLUDE
7,tourism,artwork,RETAIN
8,natural,ridge,RETAIN
9,natural,waterfall,RETAIN


In [59]:
## Joining them with the 115 original mappings
validated_scope = feature_scope.merge(
    validation_decisions_df,
    on=["osm_key", "osm_value"],
    how="left",
)

# Mappings that did not require targeted validation remain retained.
validated_scope["decision"] = (
    validated_scope["decision"]
    .fillna("RETAIN")
)

validated_scope[
    ["category", "osm_key", "osm_value", "decision"]
].head(20)

,category,osm_key,osm_value,decision
0,attractions,tourism,attraction,RETAIN
1,attractions,man_made,lighthouse,RETAIN
2,attractions,man_made,observatory,RETAIN
3,viewpoints,tourism,viewpoint,RETAIN
4,museums_culture,tourism,museum,RETAIN
5,museums_culture,tourism,gallery,RETAIN
6,museums_culture,tourism,artwork,RETAIN
7,museums_culture,amenity,theatre,RETAIN
8,history_heritage,historic,monument,RETAIN
9,history_heritage,historic,memorial,RETAIN


In [60]:
## Quick Sanity check
decision_summary = (
    validated_scope["decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="mapping_count")
)

decision_summary

,decision,mapping_count
0,RETAIN,110
1,REFINE,4
2,EXCLUDE,1


### 6.2 Refinement Rules

Mappings classified as `REFINE` remain semantically relevant to ColMaps, but
their primary OSM classification does not provide sufficient information for
unconditional inclusion.

The targeted-validation results identified four such mappings:
`natural=water`, `natural=wetland`, `leisure=swimming_pool`, and
`leisure=garden`.

Refinement rules are defined separately from the primary feature mappings so
that the distinction between source classification and additional validation
conditions remains explicit. These rules describe which secondary OSM
attributes must be considered during final dataset preparation.

The objective is not to require secondary information for every object. OSM
tag coverage is incomplete, and the absence of a secondary attribute cannot
generally be interpreted as evidence that a feature is unsuitable. Instead,
the rules identify explicit information that can be used to distinguish known
feature types or restrictions when such information is available.


In [61]:
# Define the secondary attributes identified during targeted validation
# as relevant for mappings requiring refinement.
refinement_rules = [
    {
        "osm_key": "natural",
        "osm_value": "water",
        "secondary_key": "water",
        "rule_type": "classification",
        "reason": (
            "natural=water contains multiple types of water bodies; "
            "water=* provides additional semantic classification."
        ),
    },
    {
        "osm_key": "natural",
        "osm_value": "wetland",
        "secondary_key": "wetland",
        "rule_type": "classification",
        "reason": (
            "natural=wetland contains multiple wetland environments; "
            "wetland=* provides additional semantic classification."
        ),
    },
    {
        "osm_key": "leisure",
        "osm_value": "swimming_pool",
        "secondary_key": "access",
        "rule_type": "restriction",
        "reason": (
            "leisure=swimming_pool includes facilities with different "
            "access conditions, including explicitly private pools."
        ),
    },
    {
        "osm_key": "leisure",
        "osm_value": "garden",
        "secondary_key": "access",
        "rule_type": "restriction",
        "reason": (
            "leisure=garden includes both accessible and explicitly "
            "private or restricted gardens."
        ),
    },
]

refinement_rules_df = pd.DataFrame(refinement_rules)

refinement_rules_df

,osm_key,osm_value,secondary_key,rule_type,reason
0,natural,water,water,classification,natural=water contains multiple types of water...
1,natural,wetland,wetland,classification,natural=wetland contains multiple wetland envi...
2,leisure,swimming_pool,access,restriction,leisure=swimming_pool includes facilities with...
3,leisure,garden,access,restriction,leisure=garden includes both accessible and ex...


#### Classification and restriction refinements

The refinement cases represent two different types of validation requirements.

For `leisure=swimming_pool` and `leisure=garden`, the targeted analysis
identified explicit access restrictions. Objects tagged with `access=private`
or `access=no` provide direct evidence that they should not be treated as
generally accessible traveler destinations. Missing access information is not
treated as an exclusion condition because `access=*` coverage is limited and
its absence does not imply restricted accessibility.

For `natural=water` and `natural=wetland`, the secondary attributes serve a
different purpose. The observed `water=*` and `wetland=*` values demonstrate
that the primary mappings contain heterogeneous geographic environments.
However, the profiling results alone do not establish that every individual
secondary classification should be universally included or excluded.

Consequently, these secondary classifications are retained as explicit
refinement attributes rather than converted into arbitrary allowlists based
only on occurrence frequency. Their values can be preserved during dataset
preparation so that application-level relevance rules can distinguish different
natural feature types without treating the broad primary classifications as
semantically uniform.

### 6.3 Validated Scope Export

The consolidated validation decisions are exported as a versioned feature-scope
specification for the subsequent dataset-preparation stage.

The validated specification preserves the original ColMaps category and OSM
mapping information while adding the validation decision associated with each
mapping. For mappings requiring refinement, the relevant secondary OSM
attribute and refinement type are also recorded.

This separates the semantic validation performed in this notebook from the
deterministic filtering operations implemented by the subsequent data-
preparation pipeline.

The resulting specification is stored as
`filters/02_validated_feature_scope.csv`.


In [62]:
# Add refinement metadata to the validated feature scope.
validated_scope = validated_scope.merge(
    refinement_rules_df[
        [
            "osm_key",
            "osm_value",
            "secondary_key",
            "rule_type",
        ]
    ],
    on=["osm_key", "osm_value"],
    how="left",
)

validated_scope[
    [
        "category",
        "osm_key",
        "osm_value",
        "decision",
        "secondary_key",
        "rule_type",
    ]
].query("decision != 'RETAIN'")

,category,osm_key,osm_value,decision,secondary_key,rule_type
26,nature,natural,water,REFINE,water,classification
27,nature,natural,wetland,REFINE,wetland,classification
60,parks_recreation,leisure,garden,REFINE,access,restriction
65,parks_recreation,leisure,swimming_pool,REFINE,access,restriction
112,transport_travel,amenity,parking_entrance,EXCLUDE,NaN,NaN


In [63]:
# Encode explicit exclusion values established during validation.
validated_scope["excluded_values"] = pd.NA

access_refinement_mask = (
    (validated_scope["decision"] == "REFINE")
    & (validated_scope["secondary_key"] == "access")
)

validated_scope.loc[
    access_refinement_mask,
    "excluded_values"
] = "private|no"

In [64]:
validated_scope[
    [
        "category",
        "osm_key",
        "osm_value",
        "decision",
        "secondary_key",
        "rule_type",
        "excluded_values",
    ]
].query("decision != 'RETAIN'")

,category,osm_key,osm_value,decision,secondary_key,rule_type,excluded_values
26,nature,natural,water,REFINE,water,classification,<NA>
27,nature,natural,wetland,REFINE,wetland,classification,<NA>
60,parks_recreation,leisure,garden,REFINE,access,restriction,private|no
65,parks_recreation,leisure,swimming_pool,REFINE,access,restriction,private|no
112,transport_travel,amenity,parking_entrance,EXCLUDE,NaN,NaN,<NA>


In [65]:
## Sanitty Check 
# Validate the final scope before exporting it.
assert len(validated_scope) == len(feature_scope), (
    "The validated scope must preserve the number of original mappings."
)

assert validated_scope[
    ["osm_key", "osm_value"]
].duplicated().sum() == 0, (
    "Duplicate OSM key/value mappings were found."
)

assert set(validated_scope["decision"].unique()) <= {
    "RETAIN",
    "REFINE",
    "EXCLUDE",
}, "Unexpected validation decision found."

assert (validated_scope["decision"] == "REFINE").sum() == 4
assert (validated_scope["decision"] == "EXCLUDE").sum() == 1
assert (validated_scope["decision"] == "RETAIN").sum() == 110

print("Validated scope checks passed.")

Validated scope checks passed.


In [66]:
## Exporting validate scope
VALIDATED_SCOPE_PATH = Path(
    "../filters/02_validated_feature_scope.csv"
)

validated_scope.to_csv(
    VALIDATED_SCOPE_PATH,
    index=False,
)

print(f"Validated scope exported to: {VALIDATED_SCOPE_PATH}")
print(f"Mappings exported: {len(validated_scope):,}")

Validated scope exported to: ..\filters\02_validated_feature_scope.csv
Mappings exported: 115


### 6.4 Final Observations

The feature-scope validation evaluated the 115 candidate OSM mappings defined
for the 12 ColMaps categories. Rather than manually reviewing every mapping,
targeted validation focused on mappings presenting semantic, descriptive,
spatial, representational, or classification-related uncertainty.

Of the 115 candidate mappings:

* 110 were retained without additional semantic filtering requirements.
* 4 were retained with refinement requirements.
* 1 was excluded as an independent ColMaps feature.

The refined mappings represent two different requirements.
`natural=water` and `natural=wetland` require preservation of their secondary
`water=*` and `wetland=*` classifications so that heterogeneous natural
features are not treated as semantically equivalent.
`leisure=swimming_pool` and `leisure=garden` require consideration of explicit
access restrictions, with `access=private` and `access=no` excluded from the
generally accessible destination scope.

`amenity=parking_entrance` was excluded as an independent traveler service
because it primarily represents access infrastructure rather than the parking
facility itself. The existing `amenity=parking` mapping remains responsible for
representing parking as a traveler service.

The validation also confirmed that linear and polygonal OSM representations do
not by themselves justify removing otherwise relevant features. Features such
as cliffs, ridges, and bare-rock areas therefore retain their original
geometries, while representative locations may be derived later when required
for visualization, recommendation, or route-proximity operations.

Both `natural=waterfall` and `waterway=waterfall` were retained. Although direct
tag overlap exists, the observed populations are not interchangeable, and
removing either convention could discard independently represented waterfall
features. Duplicate classifications on the same OSM object can instead be
consolidated during subsequent processing.

The final output of this notebook is
`filters/02_validated_feature_scope.csv`. This specification records the
validated semantic scope and its refinement requirements without performing
the final production filtering of the original Colombia dataset.

The subsequent data-preparation stage uses this validated specification
together with the original `.osm.pbf` source to construct the deterministic
dataset intended for import into PostGIS.


In [67]:
# Display the final validation summary.
final_summary = (
    validated_scope["decision"]
    .value_counts()
    .reindex(["RETAIN", "REFINE", "EXCLUDE"], fill_value=0)
    .rename_axis("decision")
    .reset_index(name="mapping_count")
)

display(final_summary)

print(f"Total validated mappings: {len(validated_scope):,}")
print(f"Output: {VALIDATED_SCOPE_PATH}")

,decision,mapping_count
0,RETAIN,110
1,REFINE,4
2,EXCLUDE,1


Total validated mappings: 115
Output: ..\filters\02_validated_feature_scope.csv
